# Chapitre 9 - Exercices 9.4 à 9.6 : backfitting dégénéré, degrés de liberté d'un arbre, données ozone

- **Exercice 9.4** : backfitting avec deux fois le même lisseur ; limite des résidus,
  monotonie de la somme des carrés.
- **Exercice 9.5** : degrés de liberté d'un arbre de régression, par simulation et par
  une matrice de lissage approchée.
- **Exercice 9.6** : modèle additif, arbre, MARS et PRIM sur les données ozone,
  comparés à l'affichage conditionnel de la figure 6.9.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch09_arbres import (SplineLisseur, MatriceLisseur, backfitting, ArbreCART,
                             cart_cv, MARS, GAM, prim, prim_peel, in_box, box_rules)

set_style()
rng = np.random.RandomState(0)

## Exercice 9.4 : Backfitting avec deux termes identiques

**Énoncé (reformulé).** On utilise le **même** lisseur $S$ pour estimer les deux termes
d'un modèle additif à deux termes (les deux variables sont identiques). On suppose $S$
symétrique, à valeurs propres dans $[0,1)$.

1. Montrer que le résidu du backfitting converge vers $(I+S)^{-1}(I-S)\mathbf y$ et que
   la somme des carrés résiduelle converge **en croissant**.
2. La somme des carrés résiduelle peut-elle croître dans des situations moins
   structurées ?
3. Comparer cet ajustement à celui d'un seul terme lissé par $S$. (Indication : utiliser
   la décomposition spectrale de $S$.)

### Rappel : décomposition spectrale

Une matrice réelle symétrique $S$ ($N\times N$) s'écrit $S = UDU^T$ avec $U$ orthogonale
($U^TU=I$, colonnes $u_1,\dots,u_N$ orthonormées) et $D=\mathrm{diag}(d_1,\dots,d_N)$
(théorème spectral). Écrivons $\mathbf y=\sum_k c_ku_k$ avec $c_k=u_k^T\mathbf y$. Toute
matrice de la forme $g(S)$ (polynôme ou fraction rationnelle en $S$) agit
**composante par composante** : $g(S)\mathbf y = \sum_k g(d_k)c_ku_k$. Ici
$d_k\in[0,1)$.

### 1. Limite du résidu

On part de $\mathbf f_1^{(0)}=\mathbf f_2^{(0)}=0$ (initialisation standard ; on omet
la constante en supposant $\mathbf y$ centrée). Le cycle $m\ge1$ est
$$\mathbf f_1^{(m)} = S(\mathbf y-\mathbf f_2^{(m-1)}),\qquad \mathbf f_2^{(m)} =
S(\mathbf y-\mathbf f_1^{(m)}).$$
Tous les itérés sont des fonctions de $S$ appliquées à $\mathbf y$ : on peut raisonner
sur **une composante** $k$. Notons $d=d_k$, $c=c_k$, et $a_m$, $b_m$ les composantes de
$\mathbf f_1^{(m)}$, $\mathbf f_2^{(m)}$ sur $u_k$ :
$$a_m = d(c-b_{m-1}),\qquad b_m = d(c-a_m),\qquad b_0=0 .$$

En substituant : $b_m = dc - d^2c + d^2b_{m-1}$. Le point fixe $b^*$ vérifie
$b^*(1-d^2) = dc(1-d)$, soit $b^* = \dfrac{dc}{1+d}$, et
$$b_m - b^* = d^2(b_{m-1}-b^*) \quad\Rightarrow\quad b_m = b^*\big(1-d^{2m}\big)
\xrightarrow[m\to\infty]{} b^* \quad (0\le d<1).$$
De même $a_m = dc - db_{m-1} \to dc-db^* = \dfrac{dc}{1+d} = b^*$. La composante du
résidu $\mathbf y-\mathbf f_1-\mathbf f_2$ tend donc vers
$$c - \frac{2dc}{1+d} = \frac{1-d}{1+d}\,c ,$$
ce qui, recomposé sur toutes les composantes, s'écrit
$$\mathbf r^{(m)}\longrightarrow (I+S)^{-1}(I-S)\,\mathbf y .\qquad\blacksquare$$

### La somme des carrés converge en croissant

Calculons exactement la composante du résidu après le cycle $m$. Avec $a_m = dc -
db_{m-1} = b^* + b^*d^{2m-1}$ (car $dc-db^* = b^*$ et $db^*d^{2m-2} = b^*d^{2m-1}$) et
$b_m = b^*-b^*d^{2m}$ :
$$\rho_m := c - a_m - b_m = (c-2b^*) - b^*d^{2m-1}(1-d) = \frac{1-d}{1+d}c -
\frac{dc}{1+d}\,d^{2m-1}(1-d) = \frac{(1-d)c}{1+d}\big(1-d^{2m}\big).$$
Ainsi $|\rho_m| = \frac{1-d}{1+d}|c|\,(1-d^{2m})$ est **croissante** en $m$ et tend vers
$\frac{1-d}{1+d}|c|$. Par l'orthonormalité de $U$, $\mathrm{RSS}_m = \|\mathbf
r^{(m)}\|^2 = \sum_k\rho_{m,k}^2$ est une somme de termes croissants : la somme des
carrés résiduelle **croît** vers sa limite $\|(I+S)^{-1}(I-S)\mathbf y\|^2$.

**Vérification numérique.** On prend pour $S$ une spline de lissage avec un petit terme
de ridge (valeurs propres dans $[0,1)$).

In [2]:
N = 60
x = np.sort(rng.uniform(0, 1, N))
y = np.sin(6 * x) + 0.4 * rng.randn(N)
y = y - y.mean()
S = SplineLisseur(x, df=6, ridge=0.02).matrix()
I = np.eye(N)
ev = np.linalg.eigvalsh(S)
print("valeurs propres de S dans [%.3f, %.3f]" % (ev.min(), ev.max()))
_, F, it, rss = backfitting(y, [MatriceLisseur(S), MatriceLisseur(S)], intercept=False,
                            tol=1e-13, max_iter=5000, history=True)
r_lim = np.linalg.solve(I + S, (I - S) @ y)
print("cycles : %d ; écart du résidu final à (I+S)^-1 (I-S) y : %.2e"
      % (it, np.abs(y - F.sum(1) - r_lim).max()))
print("RSS croissante à chaque cycle :", np.all(np.diff(rss) >= -1e-12))

valeurs propres de S dans [0.000, 0.980]
cycles : 613 ; écart du résidu final à (I+S)^-1 (I-S) y : 2.13e-14
RSS croissante à chaque cycle : True
